# Gossip Protocols — the numbers behind the post

Epidemic spread, message costs, SWIM false positives, and phi accrual, all recomputed. Sources: Demers et al. 1987 (epidemic algorithms), Das/Gupta/Motivala 2002 (SWIM), Hayashibara et al. 2004 (phi accrual), HashiCorp memberlist/Lifeguard 2017.

In [ ]:
# 1. Epidemic spread: how fast does a fact reach 1,000 nodes?
import numpy as np
rng = np.random.default_rng(7)
N = 1000

def spread(fanout, rounds=20):
    informed = np.zeros(N, dtype=bool); informed[0] = True
    hist = [1]
    for _ in range(rounds):
        src = np.where(informed)[0]
        targets = rng.integers(0, N, size=(len(src), fanout)).ravel()
        informed[targets] = True
        hist.append(int(informed.sum()))
        if informed.all():
            break
    return np.array(hist)

for f in (1, 2, 3):
    h = spread(f)
    r99 = int(np.argmax(h >= 0.99 * N)) if (h >= 0.99 * N).any() else len(h) - 1
    r100 = len(h) - 1
    print(f"fanout {f}: 99% at round {r99}, 100% at round {r100}")
# log_3(1000) = 6.3 rounds of pure tripling; the straggler tail adds ~1-3 rounds.
assert int(np.argmax(spread(3) >= 0.99 * N)) <= 8

In [ ]:
# 2. Message cost: all-to-all heartbeats vs gossip
for n in (10, 100, 1000, 10000):
    alltoall = n * (n - 1)
    gossip = 3 * n
    print(f"N={n:6d}: all-to-all {alltoall:10,} msgs/round, gossip {gossip:7,} "
          f"msgs/round, {alltoall / gossip:8.0f}x fewer")
assert 1000 * 999 / (3 * 1000) == 333.0  # the 1,000-node figure in the post

In [ ]:
# 3. SWIM: what indirect probes do to the false-positive rate
# Direct ping alone, independent packet-loss probability p per probe: P(false suspect) = p.
# With K indirect probes, a live node is suspected only if all K+1 probes are lost: p**(K+1).
p, K = 0.01, 3
print(f"packet loss p={p}: direct-only false suspect {p:.0%}")
print(f"with K={K} indirect probes: {p ** (K + 1):.2e} per round "
      f"({p / p ** (K + 1):,.0f}x reduction)")
# Detection latency: probe period 1 s + K parallel indirect probes (~1 s) + suspicion
# timeout T=5 s -> worst case ~7 s from death to cluster-wide 'dead'.
print("worst-case detection latency: 1s probe + 1s indirect + 5s suspicion timeout = ~7s")
assert p ** (K + 1) == 1e-8

In [ ]:
# 4. Phi accrual: suspicion from heartbeat history
# Heartbeats every mu=1.0 s, sd=0.1 s (normal). phi(t) = -log10 P(T > t).
from math import erf, sqrt, log10

def phi(gap, mu=1.0, sd=0.1):
    z = (gap - mu) / (sd * sqrt(2))
    p_late = 0.5 * (1 - erf(z))
    return -log10(max(p_late, 1e-300))

for gap in (1.0, 1.2, 1.5, 1.6, 2.0):
    print(f"gap {gap:.1f}s -> phi = {phi(gap):6.2f}")
# Usual threshold phi=8: crossed between 1.5 s and 1.6 s for this steady node.
assert phi(1.0) < 0.5 and 1.0 < phi(1.2) < 2.0 and 5 < phi(1.5) < 8 and phi(1.6) > 8
# A jittery node (sd=0.5 s) needs a much longer silence to reach the same suspicion:
print(f"jittery node (sd=0.5): gap 2.0s -> phi = {phi(2.0, sd=0.5):.2f} (vs "
      f"{phi(2.0, sd=0.1):.0f} for the steady node)")
assert phi(2.0, sd=0.5) < 2.0